# Induced Sparsity Experiemnts
This colab shows the code setup we used for the l1 regularization experiments discussed in section 4.1 of our paper.

#Setup
Code for imports, model setup, and training

#

In [ ]:
import torch, itertools, numpy as np
import torch.nn as nn, torch.optim as optim
from scipy.stats import pearsonr
import pandas as pd
import seaborn as sns, matplotlib.pyplot as plt



class TinyMLP(nn.Module):
    """
    # Simple MLP for XOR:
    # 2 inputs → hidden layer (ReLU) → 1 output (sigmoid)
    """

    def __init__(self, in_dim=2, hidden_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        return torch.sigmoid(self.fc2(torch.relu(self.fc1(x))))

def make_xor_data():

    """
    Generates XOR Dataset
    """
    X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
    y = torch.tensor([[0.],[1.],[1.],[0.]])
    return X, y

def train_model(model, X, y, l1=0.0, lr=0.05, epochs=5000):
    """
    Trains model using Adam optimizer and binary cross-entropy loss.
    Applies optional L1 regularization on the first layer weights (fc1) to encourage sparsity
    """
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    for _ in range(epochs):
        opt.zero_grad()
        y_pred = model(X)
        bce = loss_fn(y_pred, y)
        l1_term = model.fc1.weight.abs().sum() * l1
        (bce + l1_term).backward()
        opt.step()
    return model


# Experiment setup

Code for evaluating and comparing generated circuits

In [ ]:
def functional_similarity(model, circuit, X, ref_outputs):
    """
    Measures how well a given neuron subset ("circuit") reproduces
    the model's original outputs.

    Process:
    - Creates a mask based off of circuit to keep only select neurons active
    - Uses a hook to zero out anything not part of the circuit
    - Computes model output for only the circuit
    - Calculates Normalized Similarity and Pearson Correlation with reference output

    """
    with torch.no_grad():
        mask = torch.zeros(model.fc1.out_features)
        mask[list(circuit)] = 1.0
        def hook(module, inp, out): return out * mask
        h = model.fc1.register_forward_hook(hook)
        out = model(X)
        h.remove()
        mse = torch.mean((out - ref_outputs)**2).item()
        var_ref = torch.var(ref_outputs).item() + 1e-8
        norm_sim = 1 - mse / var_ref
        corr, _ = pearsonr(out.flatten().numpy(), ref_outputs.flatten().numpy())
    return max(0, min(1, norm_sim)), corr

def circuit_overlap(circuits):
    """
    Computes mean overlap for circuit
    """

    overlaps = []
    for i in range(len(circuits)):
        for j in range(i+1, len(circuits)):
            A,B = set(circuits[i]), set(circuits[j])
            overlaps.append(len(A & B)/len(A | B))
    return np.mean(overlaps) if overlaps else 0.0

def evaluate_circuit_causal(model, circuit, X, ref_outputs):
    """
    Evaluates causal role of circuits

    Sufficiency - how well the circuit reproduces output
    Necessity - how much performance drops without circuit
    """
    with torch.no_grad():
        mask = torch.zeros(model.fc1.out_features)
        mask[list(circuit)] = 1.0
        def hook_suff(m, i, o): return o * mask
        h1 = model.fc1.register_forward_hook(hook_suff)
        suff_out = model(X); h1.remove()
        suff_mse = torch.mean((suff_out - ref_outputs)**2).item()

        def hook_nec(m, i, o): return o * (1 - mask)
        h2 = model.fc1.register_forward_hook(hook_nec)
        nec_out = model(X); h2.remove()
        nec_mse = torch.mean((nec_out - ref_outputs)**2).item()
    return suff_mse, nec_mse

def enumerate_circuits(model, X):
    with torch.no_grad():
        ref_outputs = model(X)
    hidden_dim = model.fc1.out_features
    all_circuits = []
    for k in range(1, hidden_dim+1):
        for subset in itertools.combinations(range(hidden_dim), k):
            all_circuits.append(subset)
    results = []
    for c in all_circuits:
        sim, corr = functional_similarity(model, c, X, ref_outputs)
        results.append({"circuit": c, "similarity": sim, "corr": corr})
    return pd.DataFrame(results)

# Running the Experiment



In [ ]:
"""
Setup for running experiment and gathering data

Runs trials with MLPs of varying diameters and l1 regularization
Gathers data from those models in dataframe

Data:
- Seed (keep results consistent)
- Hidden Diameter
- L1 Regularization
- Num of Valid Circuits
- Mean Overlap
- Mean Similarity

"""

X, y = make_xor_data()
records = []
for seed in range(5):
    torch.manual_seed(seed)
    for hidden_dim in [3,4,5,6]:
        for l1 in [0.0, 1e-4, 1e-3]:
            model = TinyMLP(2, hidden_dim)
            model = train_model(model, X, y, l1=l1)
            results = enumerate_circuits(model, X)
            valid = results[results["similarity"] >= 0.9]
            valid_circuits = [tuple(c) for c in valid.circuit]
            mean_overlap = circuit_overlap(valid_circuits)
            records.append({
                "seed": seed,
                "hidden_dim": hidden_dim,
                "l1": l1,
                "num_valid": len(valid_circuits),
                "mean_overlap": mean_overlap,
                "mean_similarity": valid.similarity.mean()
            })
df = pd.DataFrame(records)

#Displays L1 Regularization data
sns.scatterplot(data=df, x="hidden_dim", y="num_valid", hue="l1")
plt.show()

NameError: name 'make_xor_data' is not defined